In [1]:
import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd

from conv_wm.config import load_config

In [3]:
ffprobe_path = shutil.which("ffprobe")
ffprobe_path

'/opt/homebrew/opt/ffmpeg@7/bin/ffprobe'

In [4]:
cfg = load_config()

raw_root = Path(cfg.paths.raw)

manifest_path = Path(cfg.paths.reports) / "manifest" / "raw_manifest.parquet"

manifest = pd.read_parquet(manifest_path)

videos = manifest[manifest["file_type"].eq("video")].copy()

videos.groupby("dataset").size()

dataset
ego4d     194
egocom    175
dtype: int64

In [5]:
videos.head()

,dataset,relative_path,file_name,extension,file_type,size_bytes,checksum,checksum_algorithm
7,ego4d,Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-b...,004a1802-c546-4dcc-86ba-bf1080077017.mp4,.mp4,video,353137387,<NA>,<NA>
8,ego4d,Ego4D/v2/video_540ss/035eb249-9923-41ec-9f59-3...,035eb249-9923-41ec-9f59-3b131c12bb1f.mp4,.mp4,video,283357120,<NA>,<NA>
9,ego4d,Ego4D/v2/video_540ss/03e90bbc-7d6b-423c-84d9-b...,03e90bbc-7d6b-423c-84d9-b5be3eff11c5.mp4,.mp4,video,379945894,<NA>,<NA>
10,ego4d,Ego4D/v2/video_540ss/0518d285-b7b0-4f98-ae06-a...,0518d285-b7b0-4f98-ae06-a95e0248ccfc.mp4,.mp4,video,376433443,<NA>,<NA>
11,ego4d,Ego4D/v2/video_540ss/0538719e-78e5-45dd-a811-f...,0538719e-78e5-45dd-a811-f7d32ce1d02b.mp4,.mp4,video,521996588,<NA>,<NA>


In [6]:
samples = videos.groupby("dataset", group_keys=False).head(1)[
    ["dataset", "relative_path"]
]

samples

,dataset,relative_path
7,ego4d,Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-b...
203,egocom,EgoCom/240p/20min/vid_088__day_2__con_3__perso...


In [7]:
sample_paths = {
    row.dataset: raw_root / row.relative_path for row in samples.itertuples()
}

sample_paths

{'ego4d': PosixPath('/Volumes/PortableSSD/data/raw/Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-bf1080077017.mp4'),
 'egocom': PosixPath('/Volumes/PortableSSD/data/raw/EgoCom/240p/20min/vid_088__day_2__con_3__person_1.MP4')}

In [8]:
for dataset, path in sample_paths.items():
    print(dataset, path.exists(), path)

ego4d True /Volumes/PortableSSD/data/raw/Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-bf1080077017.mp4
egocom True /Volumes/PortableSSD/data/raw/EgoCom/240p/20min/vid_088__day_2__con_3__person_1.MP4


In [9]:
def ffprobe(path: Path) -> dict:
    command = [
        "ffprobe",
        "-v",
        "error",
        "-show_format",
        "-show_streams",
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )

    return json.loads(result.stdout)

In [10]:
probes = {dataset: ffprobe(path) for dataset, path in sample_paths.items()}

In [11]:
for dataset, probe in probes.items():
    print("\n", dataset.upper())
    print("top-level:", probe.keys())
    print("number of streams:", len(probe["streams"]))

    for stream in probe["streams"]:
        print(
            "stream",
            stream.get("index"),
            stream.get("codec_type"),
            stream.get("codec_name"),
        )


 EGO4D
top-level: dict_keys(['streams', 'format'])
number of streams: 2
stream 0 video vp9
stream 1 audio aac

 EGOCOM
top-level: dict_keys(['streams', 'format'])
number of streams: 2
stream 0 video h264
stream 1 audio aac


In [12]:
def inspect_probe(probe: dict) -> None:
    format_info = probe["format"]

    print("FORMAT")
    for key in [
        "format_name",
        "start_time",
        "duration",
        "size",
        "bit_rate",
    ]:
        print(f"  {key}: {format_info.get(key)}")

    for stream in probe["streams"]:
        stream_type = stream.get("codec_type")

        print(f"\n{stream_type.upper()} STREAM")

        if stream_type == "video":
            keys = [
                "index",
                "codec_name",
                "width",
                "height",
                "pix_fmt",
                "r_frame_rate",
                "avg_frame_rate",
                "time_base",
                "start_pts",
                "start_time",
                "duration_ts",
                "duration",
                "nb_frames",
            ]

        elif stream_type == "audio":
            keys = [
                "index",
                "codec_name",
                "sample_rate",
                "channels",
                "channel_layout",
                "time_base",
                "start_pts",
                "start_time",
                "duration_ts",
                "duration",
                "nb_frames",
            ]

        else:
            keys = [
                "index",
                "codec_name",
                "time_base",
                "start_time",
                "duration",
            ]

        for key in keys:
            print(f"  {key}: {stream.get(key)}")

In [13]:
for dataset, probe in probes.items():
    print("=" * 70)
    print(dataset.upper())
    print("=" * 70)

    inspect_probe(probe)

    print()

EGO4D
FORMAT
  format_name: mov,mp4,m4a,3gp,3g2,mj2
  start_time: 0.000000
  duration: 1447.317000
  size: 353137387
  bit_rate: 1951955

VIDEO STREAM
  index: 0
  codec_name: vp9
  width: 960
  height: 540
  pix_fmt: yuv420p
  r_frame_rate: 30/1
  avg_frame_rate: 30/1
  time_base: 1/15360
  start_pts: 0
  start_time: 0.000000
  duration_ts: 22223360
  duration: 1446.833333
  nb_frames: 43405

AUDIO STREAM
  index: 1
  codec_name: aac
  sample_rate: 48000
  channels: 1
  channel_layout: mono
  time_base: 1/48000
  start_pts: 0
  start_time: 0.000000
  duration_ts: 69471216
  duration: 1447.317000
  nb_frames: 67845

EGOCOM
FORMAT
  format_name: mov,mp4,m4a,3gp,3g2,mj2
  start_time: 0.000000
  duration: 1233.500000
  size: 83615839
  bit_rate: 542299

VIDEO STREAM
  index: 0
  codec_name: h264
  width: 352
  height: 240
  pix_fmt: yuv420p
  r_frame_rate: 30/1
  avg_frame_rate: 30/1
  time_base: 1/15360
  start_pts: 0
  start_time: 0.000000
  duration_ts: 18946560
  duration: 1233.500000

In [14]:
from fractions import Fraction


def parse_fraction(value: str | None) -> float | None:
    if value in (None, "0/0", "N/A"):
        return None

    return float(Fraction(value))


def get_stream(
    probe: dict,
    codec_type: str,
) -> dict | None:
    return next(
        (
            stream
            for stream in probe["streams"]
            if stream.get("codec_type") == codec_type
        ),
        None,
    )


def normalize_probe(
    dataset: str,
    relative_path: str,
    probe: dict,
) -> dict:
    format_info = probe["format"]

    video = get_stream(probe, "video")
    audio = get_stream(probe, "audio")

    return {
        "dataset": dataset,
        "relative_path": relative_path,
        # Container
        "container_format": format_info.get("format_name"),
        "container_duration_sec": float(format_info["duration"])
        if format_info.get("duration")
        else None,
        "container_start_time_sec": float(format_info["start_time"])
        if format_info.get("start_time")
        else None,
        "container_size_bytes": int(format_info["size"])
        if format_info.get("size")
        else None,
        "container_bit_rate": int(format_info["bit_rate"])
        if format_info.get("bit_rate")
        else None,
        # Video
        "video_present": video is not None,
        "video_codec": video.get("codec_name") if video else None,
        "video_width": video.get("width") if video else None,
        "video_height": video.get("height") if video else None,
        "video_pixel_format": video.get("pix_fmt") if video else None,
        "video_r_frame_rate": parse_fraction(video.get("r_frame_rate"))
        if video
        else None,
        "video_avg_frame_rate": parse_fraction(video.get("avg_frame_rate"))
        if video
        else None,
        "video_time_base": video.get("time_base") if video else None,
        "video_start_pts": video.get("start_pts") if video else None,
        "video_start_time_sec": float(video["start_time"])
        if video and video.get("start_time")
        else None,
        "video_duration_sec": float(video["duration"])
        if video and video.get("duration")
        else None,
        "video_nb_frames": int(video["nb_frames"])
        if video and video.get("nb_frames")
        else None,
        # Audio
        "audio_present": audio is not None,
        "audio_codec": audio.get("codec_name") if audio else None,
        "audio_sample_rate_hz": int(audio["sample_rate"])
        if audio and audio.get("sample_rate")
        else None,
        "audio_channels": audio.get("channels") if audio else None,
        "audio_channel_layout": audio.get("channel_layout") if audio else None,
        "audio_time_base": audio.get("time_base") if audio else None,
        "audio_start_pts": audio.get("start_pts") if audio else None,
        "audio_start_time_sec": float(audio["start_time"])
        if audio and audio.get("start_time")
        else None,
        "audio_duration_sec": float(audio["duration"])
        if audio and audio.get("duration")
        else None,
    }

In [15]:
rows = []

for row in samples.itertuples():
    rows.append(
        normalize_probe(
            dataset=row.dataset,
            relative_path=row.relative_path,
            probe=probes[row.dataset],
        )
    )

sample_metadata = pd.DataFrame(rows)

sample_metadata.T

,0,1
dataset,ego4d,egocom
relative_path,Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-b...,EgoCom/240p/20min/vid_088__day_2__con_3__perso...
container_format,"mov,mp4,m4a,3gp,3g2,mj2","mov,mp4,m4a,3gp,3g2,mj2"
container_duration_sec,1447.317,1233.5
container_start_time_sec,0.0,0.0
container_size_bytes,353137387,83615839
container_bit_rate,1951955,542299
video_present,True,True
video_codec,vp9,h264
video_width,960,352


In [17]:
from tqdm.auto import tqdm


def safe_ffprobe(path: Path) -> tuple[dict | None, str | None]:
    command = [
        "ffprobe",
        "-v",
        "error",
        "-show_format",
        "-show_streams",
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        return None, result.stderr.strip()

    try:
        return json.loads(result.stdout), None
    except json.JSONDecodeError as exc:
        return None, f"Invalid ffprobe JSON: {exc}"

/Users/grevy/dev/conv_wm/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
def probe_record(
    dataset: str,
    relative_path: str,
    path: Path,
) -> dict:
    probe, error = safe_ffprobe(path)

    if probe is None:
        return {
            "dataset": dataset,
            "relative_path": relative_path,
            "probe_ok": False,
            "probe_error": error,
        }

    record = normalize_probe(
        dataset=dataset,
        relative_path=relative_path,
        probe=probe,
    )

    streams = probe.get("streams", [])

    record.update(
        {
            "probe_ok": True,
            "probe_error": None,
            "n_streams": len(streams),
            "n_video_streams": sum(s.get("codec_type") == "video" for s in streams),
            "n_audio_streams": sum(s.get("codec_type") == "audio" for s in streams),
        }
    )

    return record

In [22]:
records = []

for row in tqdm(
    videos.itertuples(index=False),
    total=len(videos),
    desc="Probing media",
):
    path = raw_root / row.relative_path

    records.append(
        probe_record(
            dataset=row.dataset,
            relative_path=row.relative_path,
            path=path,
        )
    )

media_metadata = pd.DataFrame(records)

media_metadata.shape

Probing media: 100%|██████████| 369/369 [00:32<00:00, 11.24it/s]


(369, 33)

In [23]:
media_metadata["probe_ok"].value_counts(dropna=False)

probe_ok
True    369
Name: count, dtype: int64

In [24]:
(
    media_metadata.groupby(
        [
            "dataset",
            "n_video_streams",
            "n_audio_streams",
        ],
        dropna=False,
    )
    .size()
    .rename("n_files")
)

dataset  n_video_streams  n_audio_streams
ego4d    1                1                  194
egocom   1                1                  175
Name: n_files, dtype: int64

In [25]:
(
    media_metadata.groupby(
        [
            "dataset",
            "video_codec",
            "audio_codec",
        ],
        dropna=False,
    )
    .size()
    .rename("n_files")
    .sort_values(ascending=False)
)

dataset  video_codec  audio_codec
ego4d    vp9          aac            194
egocom   h264         aac            175
Name: n_files, dtype: int64

In [26]:
for column in [
    "video_avg_frame_rate",
    "video_r_frame_rate",
    "video_width",
    "video_height",
    "video_pixel_format",
]:
    print(f"\n--- {column} ---")

    display(
        media_metadata.groupby(["dataset", column], dropna=False)
        .size()
        .rename("n_files")
    )


--- video_avg_frame_rate ---


dataset  video_avg_frame_rate
ego4d    30.0                    194
egocom   30.0                    173
         60.0                      2
Name: n_files, dtype: int64


--- video_r_frame_rate ---


dataset  video_r_frame_rate
ego4d    30.0                  194
egocom   30.0                  173
         60.0                    2
Name: n_files, dtype: int64


--- video_width ---


dataset  video_width
ego4d    544             12
         960            182
egocom   352            175
Name: n_files, dtype: int64


--- video_height ---


dataset  video_height
ego4d    540             194
egocom   240             175
Name: n_files, dtype: int64


--- video_pixel_format ---


dataset  video_pixel_format
ego4d    yuv420p               194
egocom   yuv420p               175
Name: n_files, dtype: int64

In [27]:
for column in [
    "audio_sample_rate_hz",
    "audio_channels",
    "audio_channel_layout",
]:
    print(f"\n--- {column} ---")

    display(
        media_metadata.groupby(["dataset", column], dropna=False)
        .size()
        .rename("n_files")
    )


--- audio_sample_rate_hz ---


dataset  audio_sample_rate_hz
ego4d    32000                    66
         44100                    10
         48000                   118
egocom   44100                   175
Name: n_files, dtype: int64


--- audio_channels ---


dataset  audio_channels
ego4d    1                 150
         2                  44
egocom   2                 175
Name: n_files, dtype: int64


--- audio_channel_layout ---


dataset  audio_channel_layout
ego4d    mono                    150
         stereo                   44
egocom   stereo                  175
Name: n_files, dtype: int64

In [28]:
media_metadata["av_duration_delta_sec"] = (
    media_metadata["audio_duration_sec"] - media_metadata["video_duration_sec"]
)

media_metadata["container_video_delta_sec"] = (
    media_metadata["container_duration_sec"] - media_metadata["video_duration_sec"]
)

In [29]:
media_metadata.groupby("dataset")[
    [
        "container_duration_sec",
        "video_duration_sec",
        "audio_duration_sec",
        "av_duration_delta_sec",
    ]
].describe().T

dataset                             ego4d       egocom
container_duration_sec count   194.000000   175.000000
                       mean   1801.799364   792.637949
                       std    1154.215599   550.509599
                       min     299.166667     4.588005
                       25%    1235.189750   294.376995
                       50%    1443.261000  1042.506009
                       75%    1805.016667  1267.616667
                       max    7020.566667  1743.800000
video_duration_sec     count   194.000000   175.000000
                       mean   1801.623024   792.636000
                       std    1154.287018   550.509696
                       min     299.166667     4.566667
                       25%    1234.783333   294.366667
                       50%    1442.833333  1042.500000
                       75%    1805.016667  1267.616667
                       max    7020.566667  1743.800000
audio_duration_sec     count   194.000000   175.000000
                       mean   1801.752722   792.595720
                       std    1154.227923   550.527141
                       min     299.072000     4.528005
                       25%    1235.189750   294.372494
                       50%    1443.261000  1042.506009
                       75%    1804.956000  1267.544003
                       max    7020.565000  1743.771995
av_duration_delta_sec  count   194.000000   175.000000
                       mean      0.129698    -0.040280
                       std       0.273435     0.066028
                       min      -0.700667    -0.306326
                       25%      -0.102667    -0.032336
                       50%      -0.001333    -0.019342
                       75%       0.437667    -0.003674
                       max       0.589333     0.029342

In [30]:
(
    media_metadata[
        [
            "dataset",
            "relative_path",
            "video_duration_sec",
            "audio_duration_sec",
            "av_duration_delta_sec",
        ]
    ]
    .assign(abs_av_delta=lambda df: df["av_duration_delta_sec"].abs())
    .sort_values("abs_av_delta", ascending=False)
    .head(20)
)

,dataset,relative_path,video_duration_sec,audio_duration_sec,av_duration_delta_sec,abs_av_delta
44,ego4d,Ego4D/v2/video_540ss/3a53bfdc-daf7-4bbc-bb9f-a...,1888.466667,1887.766,-0.700667,0.700667
165,ego4d,Ego4D/v2/video_540ss/cebb6331-2a18-454a-862b-3...,1888.500000,1887.832,-0.668000,0.668000
83,ego4d,Ego4D/v2/video_540ss/6ec6a21e-e99c-425e-af59-6...,1902.766667,1903.356,0.589333,0.589333
121,ego4d,Ego4D/v2/video_540ss/9a47f70f-4357-42a5-87c6-2...,1391.800000,1392.388,0.588000,0.588000
91,ego4d,Ego4D/v2/video_540ss/7985aadd-1fde-41e7-b4a1-8...,1232.933333,1233.515,0.581667,0.581667
26,ego4d,Ego4D/v2/video_540ss/23ac3708-df1d-4bb0-a4c7-b...,1158.733333,1159.312,0.578667,0.578667
76,ego4d,Ego4D/v2/video_540ss/64ec6de1-0e86-49de-889c-4...,1333.966667,1334.528,0.561333,0.561333
117,ego4d,Ego4D/v2/video_540ss/95abf338-cf33-43f4-a8d0-4...,1401.133333,1401.690,0.556667,0.556667
127,ego4d,Ego4D/v2/video_540ss/a168dbf5-c18e-4e46-93c4-9...,1267.700000,1268.246,0.546000,0.546000
40,ego4d,Ego4D/v2/video_540ss/356dd07e-8165-494c-9be8-b...,1320.666667,1321.205,0.538333,0.538333


In [31]:
media_metadata.groupby("dataset")[
    [
        "container_start_time_sec",
        "video_start_time_sec",
        "audio_start_time_sec",
    ]
].describe().T

dataset                         ego4d  egocom
container_start_time_sec count  194.0   175.0
                         mean     0.0     0.0
                         std      0.0     0.0
                         min      0.0     0.0
                         25%      0.0     0.0
                         50%      0.0     0.0
                         75%      0.0     0.0
                         max      0.0     0.0
video_start_time_sec     count  194.0   175.0
                         mean     0.0     0.0
                         std      0.0     0.0
                         min      0.0     0.0
                         25%      0.0     0.0
                         50%      0.0     0.0
                         75%      0.0     0.0
                         max      0.0     0.0
audio_start_time_sec     count  194.0   175.0
                         mean     0.0     0.0
                         std      0.0     0.0
                         min      0.0     0.0
                         25%      0.0     0.0
                         50%      0.0     0.0
                         75%      0.0     0.0
                         max      0.0     0.0

In [32]:
media_metadata[
    (media_metadata["video_start_time_sec"].fillna(0) != 0)
    | (media_metadata["audio_start_time_sec"].fillna(0) != 0)
][
    [
        "dataset",
        "relative_path",
        "video_start_time_sec",
        "audio_start_time_sec",
    ]
]

,dataset,relative_path,video_start_time_sec,audio_start_time_sec


In [33]:
interesting_cols = [
    "dataset",
    "relative_path",
    "video_codec",
    "video_width",
    "video_height",
    "video_avg_frame_rate",
    "video_r_frame_rate",
    "audio_sample_rate_hz",
    "audio_channels",
    "audio_channel_layout",
    "video_duration_sec",
    "audio_duration_sec",
    "av_duration_delta_sec",
]

In [34]:
media_metadata.loc[
    (media_metadata["dataset"] == "egocom")
    & (media_metadata["video_avg_frame_rate"] != 30),
    interesting_cols,
]

,dataset,relative_path,video_codec,video_width,video_height,video_avg_frame_rate,video_r_frame_rate,audio_sample_rate_hz,audio_channels,audio_channel_layout,video_duration_sec,audio_duration_sec,av_duration_delta_sec
225,egocom,EgoCom/240p/20min/vid_119__day_4__con_1__perso...,h264,352,240,60.0,60.0,44100,2,stereo,1482.383333,1482.361995,-0.021338
277,egocom,EgoCom/240p/20min/vid_171__day_6__con_5__perso...,h264,352,240,60.0,60.0,44100,2,stereo,1276.583333,1276.586009,0.002676


In [35]:
media_metadata.loc[
    (media_metadata["dataset"] == "ego4d") & (media_metadata["video_width"] != 960),
    interesting_cols,
]

,dataset,relative_path,video_codec,video_width,video_height,video_avg_frame_rate,video_r_frame_rate,audio_sample_rate_hz,audio_channels,audio_channel_layout,video_duration_sec,audio_duration_sec,av_duration_delta_sec
21,ego4d,Ego4D/v2/video_540ss/1e83c2d1-ff03-4181-9ab5-a...,vp9,544,540,30.0,30.0,44100,1,mono,6378.566667,6378.590998,0.024331
23,ego4d,Ego4D/v2/video_540ss/2024e2b0-1ac2-455b-ae80-8...,vp9,544,540,30.0,30.0,48000,2,stereo,4021.233333,4021.227000,-0.006333
34,ego4d,Ego4D/v2/video_540ss/30294c41-c90d-438a-af19-c...,vp9,544,540,30.0,30.0,44100,1,mono,3712.100000,3712.110000,0.010000
50,ego4d,Ego4D/v2/video_540ss/47517e98-bc3a-4dbf-a251-4...,vp9,544,540,30.0,30.0,44100,1,mono,4445.866667,4445.886009,0.019342
61,ego4d,Ego4D/v2/video_540ss/566ad4e5-1ce4-4679-9d19-e...,vp9,544,540,30.0,30.0,44100,1,mono,3595.533333,3595.535011,0.001678
68,ego4d,Ego4D/v2/video_540ss/5c78b345-6201-4b55-ac1c-b...,vp9,544,540,30.0,30.0,44100,1,mono,4253.066667,4253.079002,0.012335
97,ego4d,Ego4D/v2/video_540ss/84e037a9-7bd3-4ba7-b655-5...,vp9,544,540,30.0,30.0,48000,2,stereo,4021.233333,4021.227000,-0.006333
123,ego4d,Ego4D/v2/video_540ss/9c5b7322-d1cc-4b56-ae9d-8...,vp9,544,540,30.0,30.0,44100,1,mono,3632.133333,3632.099002,-0.034331
124,ego4d,Ego4D/v2/video_540ss/9ca2dc18-2c57-44cb-8c91-4...,vp9,544,540,30.0,30.0,44100,1,mono,3594.333333,3594.285011,-0.048322
129,ego4d,Ego4D/v2/video_540ss/a223fcb2-8ffa-4826-bd0c-9...,vp9,544,540,30.0,30.0,44100,1,mono,3612.966667,3612.952993,-0.013674


In [36]:
(
    media_metadata.loc[media_metadata["dataset"] == "ego4d"]
    .groupby(
        [
            "video_width",
            "video_height",
            "video_avg_frame_rate",
            "audio_sample_rate_hz",
            "audio_channels",
            "audio_channel_layout",
        ],
        dropna=False,
    )
    .size()
    .sort_values(ascending=False)
    .rename("n_files")
)

video_width  video_height  video_avg_frame_rate  audio_sample_rate_hz  audio_channels  audio_channel_layout
960          540           30.0                  48000                 1               mono                    74
                                                 32000                 1               mono                    66
                                                 48000                 2               stereo                  42
544          540           30.0                  44100                 1               mono                    10
                                                 48000                 2               stereo                   2
Name: n_files, dtype: int64

In [37]:
duration_thresholds = [0.05, 0.1, 0.25, 0.5]

for dataset, group in media_metadata.groupby("dataset"):
    delta = group["av_duration_delta_sec"].abs()

    print(f"\n{dataset.upper()}")

    for threshold in duration_thresholds:
        print(
            f"|Δ duration| > {threshold:.2f}s:",
            int((delta > threshold).sum()),
            f"/ {len(group)}",
        )


EGO4D
|Δ duration| > 0.05s: 140 / 194
|Δ duration| > 0.10s: 131 / 194
|Δ duration| > 0.25s: 74 / 194
|Δ duration| > 0.50s: 19 / 194

EGOCOM
|Δ duration| > 0.05s: 38 / 175
|Δ duration| > 0.10s: 25 / 175
|Δ duration| > 0.25s: 3 / 175
|Δ duration| > 0.50s: 0 / 175


In [38]:
def probe_video_frames(path: Path) -> pd.DataFrame:
    command = [
        "ffprobe",
        "-v",
        "error",
        "-select_streams",
        "v:0",
        "-show_frames",
        "-show_entries",
        (
            "frame="
            "pts,"
            "pts_time,"
            "best_effort_timestamp,"
            "best_effort_timestamp_time,"
            "duration,"
            "duration_time,"
            "pkt_dts,"
            "pkt_dts_time,"
            "key_frame,"
            "pict_type"
        ),
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        check=True,
        capture_output=True,
        text=True,
    )

    payload = json.loads(result.stdout)

    return pd.DataFrame(payload.get("frames", []))

In [39]:
frames = probe_video_frames(path)

frames.head(10)

,key_frame,pts,pts_time,pkt_dts,pkt_dts_time,best_effort_timestamp,best_effort_timestamp_time,duration,duration_time,pict_type,side_data_list
0,1,0,0.000000,0.0,0.000000,0,0.000000,512,0.033333,I,[{'side_data_type': 'H.26[45] User Data Unregi...
1,0,512,0.033333,512.0,0.033333,512,0.033333,512,0.033333,B,NaN
2,0,1024,0.066667,1024.0,0.066667,1024,0.066667,512,0.033333,B,NaN
3,0,1536,0.100000,1536.0,0.100000,1536,0.100000,512,0.033333,B,NaN
4,0,2048,0.133333,2048.0,0.133333,2048,0.133333,512,0.033333,P,NaN
5,0,2560,0.166667,2560.0,0.166667,2560,0.166667,512,0.033333,B,NaN
6,0,3072,0.200000,3072.0,0.200000,3072,0.200000,512,0.033333,B,NaN
7,0,3584,0.233333,3584.0,0.233333,3584,0.233333,512,0.033333,B,NaN
8,0,4096,0.266667,4096.0,0.266667,4096,0.266667,512,0.033333,P,NaN
9,0,4608,0.300000,4608.0,0.300000,4608,0.300000,512,0.033333,B,NaN


In [40]:
frames.info()

<class 'pandas.DataFrame'>
RangeIndex: 6204 entries, 0 to 6203
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   key_frame                   6204 non-null   int64  
 1   pts                         6204 non-null   int64  
 2   pts_time                    6204 non-null   str    
 3   pkt_dts                     6202 non-null   float64
 4   pkt_dts_time                6202 non-null   str    
 5   best_effort_timestamp       6204 non-null   int64  
 6   best_effort_timestamp_time  6204 non-null   str    
 7   duration                    6204 non-null   int64  
 8   duration_time               6204 non-null   str    
 9   pict_type                   6204 non-null   str    
 10  side_data_list              1 non-null      object 
dtypes: float64(1), int64(4), object(1), str(5)
memory usage: 760.6+ KB


In [41]:
frames["timestamp_sec"] = pd.to_numeric(
    frames["best_effort_timestamp_time"],
    errors="coerce",
)

frames["delta_sec"] = frames["timestamp_sec"].diff()

In [42]:
frames[
    [
        "timestamp_sec",
        "delta_sec",
        "pts_time",
        "pkt_dts_time",
        "key_frame",
        "pict_type",
    ]
].head(20)

,timestamp_sec,delta_sec,pts_time,pkt_dts_time,key_frame,pict_type
0,0.000000,NaN,0.000000,0.000000,1,I
1,0.033333,0.033333,0.033333,0.033333,0,B
2,0.066667,0.033334,0.066667,0.066667,0,B
3,0.100000,0.033333,0.100000,0.100000,0,B
4,0.133333,0.033333,0.133333,0.133333,0,P
5,0.166667,0.033334,0.166667,0.166667,0,B
6,0.200000,0.033333,0.200000,0.200000,0,B
7,0.233333,0.033333,0.233333,0.233333,0,B
8,0.266667,0.033334,0.266667,0.266667,0,P
9,0.300000,0.033333,0.300000,0.300000,0,B


In [43]:
frames["delta_sec"].describe()

count    6.203000e+03
mean     3.333333e-02
std      4.714615e-07
min      3.333300e-02
25%      3.333300e-02
50%      3.333300e-02
75%      3.333400e-02
max      3.333400e-02
Name: delta_sec, dtype: float64

In [45]:
(frames["delta_sec"].round(6).value_counts(dropna=False).head(20))

delta_sec
0.033333    4135
0.033334    2068
NaN            1
Name: count, dtype: int64

In [46]:
expected_delta = 1 / 30

frames["delta_error_sec"] = frames["delta_sec"] - expected_delta

frames["delta_error_sec"].abs().describe()

count    6.203000e+03
mean     4.444624e-07
std      1.571538e-07
min      3.333333e-07
25%      3.333333e-07
50%      3.333333e-07
75%      6.666667e-07
max      6.666667e-07
Name: delta_error_sec, dtype: float64

In [47]:
timestamps = frames["timestamp_sec"].dropna()

print("n_frames:", len(frames))
print("n_timestamped:", len(timestamps))
print("monotonic:", timestamps.is_monotonic_increasing)
print("duplicates:", int(timestamps.duplicated().sum()))

n_frames: 6204
n_timestamped: 6204
monotonic: True
duplicates: 0


In [48]:
fps = 30.0

frames["frame_index"] = range(len(frames))

frames["naive_timestamp_sec"] = frames["frame_index"] / fps

frames["timestamp_error_sec"] = frames["timestamp_sec"] - frames["naive_timestamp_sec"]

In [49]:
frames["timestamp_error_sec"].describe()

count    6.204000e+03
mean     1.453991e-20
std      2.721875e-07
min     -3.333333e-07
25%     -3.333333e-07
50%      0.000000e+00
75%      3.333333e-07
max      3.333333e-07
Name: timestamp_error_sec, dtype: float64

In [50]:
frames["timestamp_error_sec"].abs().max()

np.float64(3.333333467026023e-07)

In [51]:
ego4d_30 = media_metadata.loc[
    (media_metadata["dataset"] == "ego4d")
    & (media_metadata["video_avg_frame_rate"] == 30)
].iloc[0]

egocom_30 = media_metadata.loc[
    (media_metadata["dataset"] == "egocom")
    & (media_metadata["video_avg_frame_rate"] == 30)
].iloc[0]

egocom_60 = media_metadata.loc[
    (media_metadata["dataset"] == "egocom")
    & (media_metadata["video_avg_frame_rate"] == 60)
].iloc[0]

In [52]:
video_samples = {
    "ego4d_30": ego4d_30,
    "egocom_30": egocom_30,
    "egocom_60": egocom_60,
}

In [53]:
def summarize_frame_timestamps(
    frames: pd.DataFrame,
    fps: float,
) -> dict:
    timestamps = pd.to_numeric(
        frames["best_effort_timestamp_time"],
        errors="coerce",
    )

    delta = timestamps.diff()

    frame_index = pd.Series(
        range(len(frames)),
        index=frames.index,
        dtype=float,
    )

    naive_timestamp = frame_index / fps

    timestamp_error = timestamps - naive_timestamp

    return {
        "n_frames": len(frames),
        "n_timestamped": int(timestamps.notna().sum()),
        "first_timestamp_sec": timestamps.iloc[0],
        "last_timestamp_sec": timestamps.iloc[-1],
        "monotonic": timestamps.is_monotonic_increasing,
        "n_duplicate_timestamps": int(timestamps.duplicated().sum()),
        "delta_min_sec": delta.min(),
        "delta_median_sec": delta.median(),
        "delta_max_sec": delta.max(),
        "n_unique_delta_rounded_6": int(delta.round(6).nunique()),
        "max_abs_naive_timestamp_error_sec": (timestamp_error.abs().max()),
    }

In [54]:
timestamp_summaries = []

for name, row in video_samples.items():
    path = raw_root / row.relative_path
    fps = float(row.video_avg_frame_rate)

    frames = probe_video_frames(path)

    summary = summarize_frame_timestamps(
        frames=frames,
        fps=fps,
    )

    timestamp_summaries.append(
        {
            "sample": name,
            "relative_path": row.relative_path,
            "fps": fps,
            **summary,
        }
    )

timestamp_summary = pd.DataFrame(timestamp_summaries)

timestamp_summary.T

,0,1,2
sample,ego4d_30,egocom_30,egocom_60
relative_path,Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-b...,EgoCom/240p/20min/vid_088__day_2__con_3__perso...,EgoCom/240p/20min/vid_119__day_4__con_1__perso...
fps,30.0,30.0,60.0
n_frames,43405,37005,88943
n_timestamped,43405,37005,88943
first_timestamp_sec,0.0,0.0,0.0
last_timestamp_sec,1446.8,1233.466667,1482.366667
monotonic,True,True,True
n_duplicate_timestamps,0,0,0
delta_min_sec,0.033333,0.033333,0.016666


In [55]:
for name, row in video_samples.items():
    path = raw_root / row.relative_path

    frames = probe_video_frames(path)

    timestamps = pd.to_numeric(
        frames["best_effort_timestamp_time"],
        errors="coerce",
    )

    print(f"\n{name}")

    display(timestamps.diff().round(6).value_counts(dropna=False).head(10))


ego4d_30


best_effort_timestamp_time
0.033333    28936
0.033334    14468
NaN             1
Name: count, dtype: int64


egocom_30


best_effort_timestamp_time
0.033333    24669
0.033334    12335
NaN             1
Name: count, dtype: int64


egocom_60


best_effort_timestamp_time
0.016667    59295
0.016666    29647
NaN             1
Name: count, dtype: int64

In [60]:
def probe_video_frames(
    path: Path,
    *,
    read_interval: str | None = None,
) -> pd.DataFrame:
    command = [
        str(ffprobe_path),
        "-v",
        "error",
        "-select_streams",
        "v:0",
    ]

    if read_interval is not None:
        command += [
            "-read_intervals",
            read_interval,
        ]

    command += [
        "-show_frames",
        "-show_entries",
        (
            "frame="
            "pts,"
            "pts_time,"
            "best_effort_timestamp,"
            "best_effort_timestamp_time,"
            "duration,"
            "duration_time,"
            "pkt_dts,"
            "pkt_dts_time,"
            "key_frame,"
            "pict_type"
        ),
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"ffprobe failed for:\n{path}\n\n"
            f"return code: {result.returncode}\n\n"
            f"stderr:\n{result.stderr}"
        )

    payload = json.loads(result.stdout)

    return pd.DataFrame(payload.get("frames", []))

In [61]:
import numpy as np
import pandas as pd

In [62]:
def fraction_to_float(value: str | float) -> float:
    if isinstance(value, str):
        return float(Fraction(value))

    return float(value)

In [63]:
def analyze_frame_timeline(
    frames: pd.DataFrame,
    *,
    fps: float,
    time_base: str,
) -> dict:
    timestamps = pd.to_numeric(
        frames["best_effort_timestamp_time"],
        errors="coerce",
    )

    valid = timestamps.dropna()

    if len(valid) < 2:
        return {
            "timeline_class": "insufficient_data",
            "n_frames": len(frames),
            "n_timestamped": len(valid),
        }

    deltas = valid.diff().dropna()

    expected_delta = 1.0 / fps
    time_base_sec = fraction_to_float(time_base)

    # Compare on a relative clock.
    relative_ts = valid - valid.iloc[0]

    frame_index = np.arange(len(valid))
    expected_ts = frame_index / fps

    drift = relative_ts.to_numpy() - expected_ts

    delta_error = deltas - expected_delta

    # Express errors in stream clock ticks.
    delta_error_ticks = delta_error.abs() / time_base_sec

    non_positive_deltas = int((deltas <= 0).sum())

    # CFR-compatible if every frame spacing differs from the expected
    # cadence by at most one stream time-base tick.
    cfr_consistent = non_positive_deltas == 0 and delta_error_ticks.max() <= 1.0 + 1e-9

    return {
        "timeline_class": ("cfr_consistent" if cfr_consistent else "vfr_or_irregular"),
        "n_frames": len(frames),
        "n_timestamped": int(valid.size),
        "timestamp_coverage": float(valid.size / len(frames)),
        "first_timestamp_sec": float(valid.iloc[0]),
        "last_timestamp_sec": float(valid.iloc[-1]),
        "monotonic": bool(valid.is_monotonic_increasing),
        "n_duplicate_timestamps": int(valid.duplicated().sum()),
        "n_non_positive_deltas": non_positive_deltas,
        "expected_delta_sec": expected_delta,
        "delta_min_sec": float(deltas.min()),
        "delta_median_sec": float(deltas.median()),
        "delta_max_sec": float(deltas.max()),
        "delta_std_sec": float(deltas.std()),
        "max_delta_error_sec": float(delta_error.abs().max()),
        "max_delta_error_ticks": float(delta_error_ticks.max()),
        "max_abs_cumulative_drift_sec": float(np.abs(drift).max()),
    }

In [66]:
timeline_results = []

for name, row in video_samples.items():
    path = raw_root / row.relative_path

    frames = probe_video_frames(path)

    result = analyze_frame_timeline(
        frames,
        fps=float(row.video_avg_frame_rate),
        time_base=row.video_time_base,
    )

    timeline_results.append(
        {
            "sample": name,
            "dataset": row.dataset,
            "relative_path": row.relative_path,
            "fps": row.video_avg_frame_rate,
            "time_base": row.video_time_base,
            **result,
        }
    )

timeline_results = pd.DataFrame(timeline_results)

timeline_results[
    [
        "sample",
        "fps",
        "timeline_class",
        "n_frames",
        "timestamp_coverage",
        "delta_median_sec",
        "delta_min_sec",
        "delta_max_sec",
        "max_delta_error_ticks",
        "max_abs_cumulative_drift_sec",
    ]
]

,sample,fps,timeline_class,n_frames,timestamp_coverage,delta_median_sec,delta_min_sec,delta_max_sec,max_delta_error_ticks,max_abs_cumulative_drift_sec
0,ego4d_30,30.0,cfr_consistent,43405,1.0,0.033333,0.033333,0.033334,0.01024,3.333334e-07
1,egocom_30,30.0,cfr_consistent,37005,1.0,0.033333,0.033333,0.033334,0.01024,3.333334e-07
2,egocom_60,60.0,cfr_consistent,88943,1.0,0.016667,0.016666,0.016667,0.01024,3.333334e-07


In [67]:
row = video_samples["ego4d_30"]
path = raw_root / row.relative_path

print(path)
print(path.exists())

test_frames = probe_video_frames(
    path,
    read_interval="0%+10",
)

test_frames.shape

/Volumes/PortableSSD/data/raw/Ego4D/v2/video_540ss/004a1802-c546-4dcc-86ba-bf1080077017.mp4
True


(300, 10)

In [ ]:
from conv_wm.data.media.video_timeline import (
    analyze_frame_timeline,
    make_timeline_windows,
    probe_video_timestamps,
)

In [71]:
def make_sample_intervals(
    duration_sec: float,
    *,
    window_sec: float = 10.0,
) -> str:
    if duration_sec <= 3 * window_sec:
        return f"0%+{duration_sec}"

    starts = [
        0.0,
        max(0.0, duration_sec / 2 - window_sec / 2),
        max(0.0, duration_sec - window_sec),
    ]

    return ",".join(f"{start:.6f}%+{window_sec:.6f}" for start in starts)

In [72]:
make_sample_intervals(1447.3)

'0.000000%+10.000000,718.650000%+10.000000,1437.300000%+10.000000'

In [73]:
import time


def benchmark_probe(
    path: Path,
    *,
    read_intervals: str | None = None,
) -> tuple[pd.DataFrame, float]:
    start = time.perf_counter()

    frames = probe_video_timestamps(
        path,
        read_intervals=read_intervals,
    )

    elapsed = time.perf_counter() - start

    return frames, elapsed

In [74]:
benchmark_samples = {
    "ego4d": media_metadata.loc[media_metadata["dataset"].eq("ego4d")].iloc[0],
    "egocom": media_metadata.loc[media_metadata["dataset"].eq("egocom")].iloc[0],
}

In [75]:
benchmark_results = []

for name, row in benchmark_samples.items():
    path = raw_root / row.relative_path
    duration = float(row.video_duration_sec)

    intervals = make_sample_intervals(duration)

    sampled_frames, sampled_time = benchmark_probe(
        path,
        read_intervals=intervals,
    )

    full_frames, full_time = benchmark_probe(path)

    benchmark_results.append(
        {
            "dataset": name,
            "duration_sec": duration,
            "sampled_frames": len(sampled_frames),
            "sampled_runtime_sec": sampled_time,
            "full_frames": len(full_frames),
            "full_runtime_sec": full_time,
            "speedup": full_time / sampled_time,
        }
    )

benchmark_results = pd.DataFrame(benchmark_results)

benchmark_results

,dataset,duration_sec,sampled_frames,sampled_runtime_sec,full_frames,full_runtime_sec,speedup
0,ego4d,1446.833333,900,1.664151,43405,46.648679,28.031516
1,egocom,1233.500000,896,0.356923,37005,8.770418,24.572281


In [77]:
def audit_sampled_video_timeline(
    row,
    *,
    raw_root: Path,
    window_sec: float = 10.0,
) -> list[dict]:
    path = raw_root / row.relative_path

    windows = make_timeline_windows(
        float(row.video_duration_sec),
        window_sec=window_sec,
    )

    results = []

    for window_name, interval in windows.items():
        try:
            frames = probe_video_timestamps(
                path,
                read_intervals=interval,
            )

            analysis = analyze_frame_timeline(
                frames,
                fps=float(row.video_avg_frame_rate),
                time_base=row.video_time_base,
            )

            results.append(
                {
                    "dataset": row.dataset,
                    "relative_path": row.relative_path,
                    "window": window_name,
                    "requested_interval": interval,
                    "probe_ok": True,
                    "probe_error": None,
                    **analysis,
                }
            )

        except Exception as exc:
            results.append(
                {
                    "dataset": row.dataset,
                    "relative_path": row.relative_path,
                    "window": window_name,
                    "requested_interval": interval,
                    "probe_ok": False,
                    "probe_error": str(exc),
                    "timeline_class": "probe_error",
                }
            )

    return results

In [78]:
from tqdm.auto import tqdm

timeline_records = []

for row in tqdm(
    media_metadata.itertuples(index=False),
    total=len(media_metadata),
    desc="Video timeline audit",
):
    timeline_records.extend(
        audit_sampled_video_timeline(
            row,
            raw_root=raw_root,
        )
    )

video_timeline_samples = pd.DataFrame(timeline_records)

Video timeline audit: 100%|██████████| 369/369 [05:24<00:00,  1.14it/s]


In [79]:
video_timeline_samples.shape

(1103, 23)

In [80]:
video_timeline_samples["probe_ok"].value_counts(dropna=False)

probe_ok
True    1103
Name: count, dtype: int64

In [81]:
(
    video_timeline_samples.groupby(
        [
            "dataset",
            "timeline_class",
        ],
        dropna=False,
    )
    .size()
    .rename("n_windows")
)

dataset  timeline_class
ego4d    cfr_consistent    582
egocom   cfr_consistent    521
Name: n_windows, dtype: int64

In [82]:
video_timeline_samples.groupby("dataset")[
    [
        "timestamp_coverage",
        "max_delta_error_ticks",
        "max_abs_cumulative_drift_sec",
    ]
].describe().T

dataset                                    ego4d        egocom
timestamp_coverage           count  5.820000e+02  5.210000e+02
                             mean   1.000000e+00  1.000000e+00
                             std    0.000000e+00  0.000000e+00
                             min    1.000000e+00  1.000000e+00
                             25%    1.000000e+00  1.000000e+00
                             50%    1.000000e+00  1.000000e+00
                             75%    1.000000e+00  1.000000e+00
                             max    1.000000e+00  1.000000e+00
max_delta_error_ticks        count  5.820000e+02  5.210000e+02
                             mean   1.024000e-02  1.024000e-02
                             std    1.602798e-09  1.059834e-09
                             min    1.024000e-02  1.024000e-02
                             25%    1.024000e-02  1.024000e-02
                             50%    1.024000e-02  1.024000e-02
                             75%    1.024000e-02  1.024000e-02
                             max    1.024001e-02  1.024000e-02
max_abs_cumulative_drift_sec count  5.820000e+02  5.210000e+02
                             mean   4.747996e-07  4.856046e-07
                             std    1.648922e-07  1.662034e-07
                             min    3.333333e-07  3.333333e-07
                             25%    3.333333e-07  3.333333e-07
                             50%    3.333335e-07  3.333334e-07
                             75%    6.666667e-07  6.666667e-07
                             max    6.666675e-07  6.666669e-07

In [83]:
file_timeline_summary = video_timeline_samples.groupby(
    ["dataset", "relative_path"],
    as_index=False,
).agg(
    n_windows=("window", "count"),
    all_probes_ok=(
        "probe_ok",
        "all",
    ),
    all_windows_cfr_consistent=(
        "timeline_class",
        lambda s: bool((s == "cfr_consistent").all()),
    ),
    min_timestamp_coverage=(
        "timestamp_coverage",
        "min",
    ),
    max_delta_error_ticks=(
        "max_delta_error_ticks",
        "max",
    ),
    max_cumulative_drift_sec=(
        "max_abs_cumulative_drift_sec",
        "max",
    ),
    total_duplicate_timestamps=(
        "n_duplicate_timestamps",
        "sum",
    ),
    total_non_positive_deltas=(
        "n_non_positive_deltas",
        "sum",
    ),
)

In [84]:
file_timeline_summary.groupby(
    [
        "dataset",
        "all_windows_cfr_consistent",
    ]
).size()

dataset  all_windows_cfr_consistent
ego4d    True                          194
egocom   True                          175
dtype: int64

In [85]:
suspects = file_timeline_summary.loc[
    ~file_timeline_summary["all_windows_cfr_consistent"]
]

suspects[
    [
        "dataset",
        "relative_path",
        "min_timestamp_coverage",
        "max_delta_error_ticks",
        "max_cumulative_drift_sec",
        "total_duplicate_timestamps",
        "total_non_positive_deltas",
    ]
]

,dataset,relative_path,min_timestamp_coverage,max_delta_error_ticks,max_cumulative_drift_sec,total_duplicate_timestamps,total_non_positive_deltas


In [86]:
# Suspects from the sampled audit.
suspect_paths = set(
    file_timeline_summary.loc[
        ~file_timeline_summary["all_windows_cfr_consistent"],
        "relative_path",
    ]
)

# Known uncommon media regimes discovered in B1.
rare_paths = set(
    media_metadata.loc[
        (
            (media_metadata["dataset"] == "egocom")
            & (media_metadata["video_avg_frame_rate"] == 60)
        )
        | (
            (media_metadata["dataset"] == "ego4d")
            & (media_metadata["video_width"] == 544)
        ),
        "relative_path",
    ]
)

mandatory_paths = suspect_paths | rare_paths

In [87]:
normal_pool = media_metadata.loc[
    ~media_metadata["relative_path"].isin(mandatory_paths)
].copy()

controls = normal_pool.groupby(
    [
        "dataset",
        "video_codec",
        "video_avg_frame_rate",
        "video_width",
        "video_height",
    ],
    group_keys=False,
).apply(
    lambda group: group.sample(
        n=min(2, len(group)),
        random_state=42,
    )
)

verification_paths = mandatory_paths | set(controls["relative_path"])

verification_set = media_metadata.loc[
    media_metadata["relative_path"].isin(verification_paths)
].copy()

verification_set[
    [
        "dataset",
        "relative_path",
        "video_codec",
        "video_avg_frame_rate",
        "video_width",
        "video_height",
    ]
]

,dataset,relative_path,video_codec,video_avg_frame_rate,video_width,video_height
19,ego4d,Ego4D/v2/video_540ss/194612d8-4baa-4e08-a382-1...,vp9,30.0,960,540
21,ego4d,Ego4D/v2/video_540ss/1e83c2d1-ff03-4181-9ab5-a...,vp9,30.0,544,540
23,ego4d,Ego4D/v2/video_540ss/2024e2b0-1ac2-455b-ae80-8...,vp9,30.0,544,540
34,ego4d,Ego4D/v2/video_540ss/30294c41-c90d-438a-af19-c...,vp9,30.0,544,540
45,ego4d,Ego4D/v2/video_540ss/3c63ce18-43fe-4ec8-a2f5-a...,vp9,30.0,960,540
50,ego4d,Ego4D/v2/video_540ss/47517e98-bc3a-4dbf-a251-4...,vp9,30.0,544,540
61,ego4d,Ego4D/v2/video_540ss/566ad4e5-1ce4-4679-9d19-e...,vp9,30.0,544,540
68,ego4d,Ego4D/v2/video_540ss/5c78b345-6201-4b55-ac1c-b...,vp9,30.0,544,540
97,ego4d,Ego4D/v2/video_540ss/84e037a9-7bd3-4ba7-b655-5...,vp9,30.0,544,540
123,ego4d,Ego4D/v2/video_540ss/9c5b7322-d1cc-4b56-ae9d-8...,vp9,30.0,544,540


In [88]:
from tqdm.auto import tqdm

full_scan_records = []

for row in tqdm(
    verification_set.itertuples(index=False),
    total=len(verification_set),
    desc="Full video timeline validation",
):
    path = raw_root / row.relative_path

    try:
        frames = probe_video_timestamps(path)

        analysis = analyze_frame_timeline(
            frames,
            fps=float(row.video_avg_frame_rate),
            time_base=row.video_time_base,
        )

        full_scan_records.append(
            {
                "dataset": row.dataset,
                "relative_path": row.relative_path,
                "video_avg_frame_rate": row.video_avg_frame_rate,
                "video_width": row.video_width,
                "video_height": row.video_height,
                "probe_ok": True,
                "probe_error": None,
                **analysis,
            }
        )

    except Exception as exc:
        full_scan_records.append(
            {
                "dataset": row.dataset,
                "relative_path": row.relative_path,
                "video_avg_frame_rate": row.video_avg_frame_rate,
                "video_width": row.video_width,
                "video_height": row.video_height,
                "probe_ok": False,
                "probe_error": str(exc),
                "timeline_class": "probe_error",
            }
        )

full_timeline_validation = pd.DataFrame(full_scan_records)

Full video timeline validation: 100%|██████████| 18/18 [17:23<00:00, 57.98s/it]


In [89]:
full_timeline_validation[
    [
        "dataset",
        "video_avg_frame_rate",
        "video_width",
        "timeline_class",
    ]
].value_counts()

dataset  video_avg_frame_rate  video_width  timeline_class
ego4d    30.0                  544          cfr_consistent    12
                               960          cfr_consistent     2
egocom   60.0                  352          cfr_consistent     2
         30.0                  352          cfr_consistent     2
Name: count, dtype: int64

In [90]:
comparison = file_timeline_summary[
    [
        "dataset",
        "relative_path",
        "all_windows_cfr_consistent",
    ]
].merge(
    full_timeline_validation[
        [
            "relative_path",
            "timeline_class",
            "max_delta_error_ticks",
            "max_abs_cumulative_drift_sec",
        ]
    ],
    on="relative_path",
    how="inner",
)

comparison

,dataset,relative_path,all_windows_cfr_consistent,timeline_class,max_delta_error_ticks,max_abs_cumulative_drift_sec
0,ego4d,Ego4D/v2/video_540ss/194612d8-4baa-4e08-a382-1...,True,cfr_consistent,0.01024,3.333334e-07
1,ego4d,Ego4D/v2/video_540ss/1e83c2d1-ff03-4181-9ab5-a...,True,cfr_consistent,0.01024,3.333334e-07
2,ego4d,Ego4D/v2/video_540ss/2024e2b0-1ac2-455b-ae80-8...,True,cfr_consistent,0.01024,3.333334e-07
3,ego4d,Ego4D/v2/video_540ss/30294c41-c90d-438a-af19-c...,True,cfr_consistent,0.01024,3.333334e-07
4,ego4d,Ego4D/v2/video_540ss/3c63ce18-43fe-4ec8-a2f5-a...,True,cfr_consistent,0.01024,3.333334e-07
5,ego4d,Ego4D/v2/video_540ss/47517e98-bc3a-4dbf-a251-4...,True,cfr_consistent,0.01024,3.333334e-07
6,ego4d,Ego4D/v2/video_540ss/566ad4e5-1ce4-4679-9d19-e...,True,cfr_consistent,0.01024,3.333334e-07
7,ego4d,Ego4D/v2/video_540ss/5c78b345-6201-4b55-ac1c-b...,True,cfr_consistent,0.01024,3.333334e-07
8,ego4d,Ego4D/v2/video_540ss/84e037a9-7bd3-4ba7-b655-5...,True,cfr_consistent,0.01024,3.333334e-07
9,ego4d,Ego4D/v2/video_540ss/9c5b7322-d1cc-4b56-ae9d-8...,True,cfr_consistent,0.01024,3.333334e-07


In [91]:
pd.crosstab(
    comparison["all_windows_cfr_consistent"],
    comparison["timeline_class"],
)

timeline_class,cfr_consistent
all_windows_cfr_consistent,
True,18


In [92]:
full_timeline_validation.loc[
    full_timeline_validation["timeline_class"] != "cfr_consistent"
][
    [
        "dataset",
        "relative_path",
        "video_avg_frame_rate",
        "timeline_class",
        "timestamp_coverage",
        "n_duplicate_timestamps",
        "n_non_positive_deltas",
        "max_delta_error_ticks",
        "max_abs_cumulative_drift_sec",
        "probe_error",
    ]
]

,dataset,relative_path,video_avg_frame_rate,timeline_class,timestamp_coverage,n_duplicate_timestamps,n_non_positive_deltas,max_delta_error_ticks,max_abs_cumulative_drift_sec,probe_error


In [93]:
from pathlib import Path

import pandas as pd


def probe_audio_frames(
    path: Path,
    *,
    read_intervals: str | None = None,
) -> pd.DataFrame:
    command = [
        str(ffprobe_path),
        "-v",
        "error",
        "-select_streams",
        "a:0",
    ]

    if read_intervals is not None:
        command += [
            "-read_intervals",
            read_intervals,
        ]

    command += [
        "-show_frames",
        "-show_entries",
        ("frame=pts,pts_time,duration,duration_time,nb_samples"),
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {path}\n{result.stderr.strip()}")

    return pd.DataFrame(json.loads(result.stdout).get("frames", []))

In [95]:
audio_samples = {
    "ego4d_32k": media_metadata.loc[
        (media_metadata["dataset"] == "ego4d")
        & (media_metadata["audio_sample_rate_hz"] == 32000)
    ].iloc[0],
    "ego4d_44k1": media_metadata.loc[
        (media_metadata["dataset"] == "ego4d")
        & (media_metadata["audio_sample_rate_hz"] == 44100)
    ].iloc[0],
    "ego4d_48k": media_metadata.loc[
        (media_metadata["dataset"] == "ego4d")
        & (media_metadata["audio_sample_rate_hz"] == 48000)
    ].iloc[0],
    "egocom_44k1": media_metadata.loc[
        (media_metadata["dataset"] == "egocom")
        & (media_metadata["audio_sample_rate_hz"] == 44100)
    ].iloc[0],
}

In [97]:
audio_results = {}

for name, row in audio_samples.items():
    path = raw_root / row.relative_path

    frames = probe_audio_frames(
        path,
        read_intervals="%+10",
    )

    audio_results[name] = frames

    print(
        name,
        "sample_rate=",
        row.audio_sample_rate_hz,
        "n_frames=",
        len(frames),
    )

    display(frames.head())

ego4d_32k sample_rate= 32000 n_frames= 311


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.032000,1024
1,1024,0.032000,1024,0.032000,1024
2,2048,0.064000,1024,0.032000,1024
3,3072,0.096000,1024,0.032000,1024
4,4096,0.128000,1024,0.032000,1024


ego4d_44k1 sample_rate= 44100 n_frames= 429


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.023220,1024
1,1024,0.023220,1024,0.023220,1024
2,2048,0.046440,1024,0.023220,1024
3,3072,0.069660,1024,0.023220,1024
4,4096,0.092880,1024,0.023220,1024


ego4d_48k sample_rate= 48000 n_frames= 467


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.021333,1024
1,1024,0.021333,1024,0.021333,1024
2,2048,0.042667,1024,0.021333,1024
3,3072,0.064000,1024,0.021333,1024
4,4096,0.085333,1024,0.021333,1024


egocom_44k1 sample_rate= 44100 n_frames= 430


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.023220,1024
1,1024,0.023220,1024,0.023220,1024
2,2048,0.046440,1024,0.023220,1024
3,3072,0.069660,1024,0.023220,1024
4,4096,0.092880,1024,0.023220,1024


In [98]:
for name, frames in audio_results.items():
    print(f"\n{name}")

    display(
        frames[
            [
                "pts",
                "pts_time",
                "duration",
                "duration_time",
                "nb_samples",
            ]
        ].head(10)
    )


ego4d_32k


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.032000,1024
1,1024,0.032000,1024,0.032000,1024
2,2048,0.064000,1024,0.032000,1024
3,3072,0.096000,1024,0.032000,1024
4,4096,0.128000,1024,0.032000,1024
5,5120,0.160000,1024,0.032000,1024
6,6144,0.192000,1024,0.032000,1024
7,7168,0.224000,1024,0.032000,1024
8,8192,0.256000,1024,0.032000,1024
9,9216,0.288000,1024,0.032000,1024



ego4d_44k1


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.023220,1024
1,1024,0.023220,1024,0.023220,1024
2,2048,0.046440,1024,0.023220,1024
3,3072,0.069660,1024,0.023220,1024
4,4096,0.092880,1024,0.023220,1024
5,5120,0.116100,1024,0.023220,1024
6,6144,0.139320,1024,0.023220,1024
7,7168,0.162540,1024,0.023220,1024
8,8192,0.185760,1024,0.023220,1024
9,9216,0.208980,1024,0.023220,1024



ego4d_48k


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.021333,1024
1,1024,0.021333,1024,0.021333,1024
2,2048,0.042667,1024,0.021333,1024
3,3072,0.064000,1024,0.021333,1024
4,4096,0.085333,1024,0.021333,1024
5,5120,0.106667,1024,0.021333,1024
6,6144,0.128000,1024,0.021333,1024
7,7168,0.149333,1024,0.021333,1024
8,8192,0.170667,1024,0.021333,1024
9,9216,0.192000,1024,0.021333,1024



egocom_44k1


,pts,pts_time,duration,duration_time,nb_samples
0,0,0.000000,1024,0.023220,1024
1,1024,0.023220,1024,0.023220,1024
2,2048,0.046440,1024,0.023220,1024
3,3072,0.069660,1024,0.023220,1024
4,4096,0.092880,1024,0.023220,1024
5,5120,0.116100,1024,0.023220,1024
6,6144,0.139320,1024,0.023220,1024
7,7168,0.162540,1024,0.023220,1024
8,8192,0.185760,1024,0.023220,1024
9,9216,0.208980,1024,0.023220,1024


In [99]:
import pandas as pd


def analyze_audio_timeline(
    frames: pd.DataFrame,
    *,
    sample_rate_hz: int,
    time_base: str,
) -> dict[str, object]:
    pts = pd.to_numeric(
        frames.get("pts", pd.Series(dtype=float)),
        errors="coerce",
    )

    nb_samples = pd.to_numeric(
        frames.get("nb_samples", pd.Series(dtype=float)),
        errors="coerce",
    )

    valid_mask = pts.notna() & nb_samples.notna()

    valid_pts = pts[valid_mask].reset_index(drop=True)
    valid_samples = nb_samples[valid_mask].reset_index(drop=True)

    if len(valid_pts) < 2:
        return {
            "timeline_class": "insufficient_data",
            "n_frames": len(frames),
            "n_valid_frames": int(valid_mask.sum()),
        }

    time_base_sec = float(Fraction(time_base))

    # Convert PTS to seconds using the stream time base.
    start_sec = valid_pts * time_base_sec

    # Exact decoded audio duration represented by each frame.
    frame_duration_sec = valid_samples / sample_rate_hz

    expected_next_start_sec = (
        start_sec.iloc[:-1].to_numpy() + frame_duration_sec.iloc[:-1].to_numpy()
    )

    actual_next_start_sec = start_sec.iloc[1:].to_numpy()

    gap_sec = actual_next_start_sec - expected_next_start_sec

    # Audio-native unit: samples.
    gap_samples = gap_sec * sample_rate_hz

    tolerance_samples = 1.0

    n_gaps = int((gap_samples > tolerance_samples).sum())

    n_overlaps = int((gap_samples < -tolerance_samples).sum())

    continuous = (
        n_gaps == 0
        and n_overlaps == 0
        and pts.notna().all()
        and nb_samples.notna().all()
    )

    return {
        "timeline_class": ("continuous" if continuous else "gap_or_overlap"),
        "n_frames": len(frames),
        "n_valid_frames": int(valid_mask.sum()),
        "timestamp_coverage": float(pts.notna().mean()),
        "sample_count_coverage": float(nb_samples.notna().mean()),
        "first_pts": int(valid_pts.iloc[0]),
        "last_pts": int(valid_pts.iloc[-1]),
        "median_nb_samples": float(valid_samples.median()),
        "min_gap_samples": float(gap_samples.min()),
        "max_gap_samples": float(gap_samples.max()),
        "max_abs_gap_samples": float(np.abs(gap_samples).max()),
        "n_gaps": n_gaps,
        "n_overlaps": n_overlaps,
    }

In [100]:
audio_timeline_results = []

for name, row in audio_samples.items():
    result = analyze_audio_timeline(
        audio_results[name],
        sample_rate_hz=int(row.audio_sample_rate_hz),
        time_base=row.audio_time_base,
    )

    audio_timeline_results.append(
        {
            "sample": name,
            "dataset": row.dataset,
            "sample_rate_hz": (row.audio_sample_rate_hz),
            "time_base": row.audio_time_base,
            **result,
        }
    )

audio_timeline_results = pd.DataFrame(audio_timeline_results)

audio_timeline_results[
    [
        "sample",
        "sample_rate_hz",
        "timeline_class",
        "timestamp_coverage",
        "median_nb_samples",
        "max_abs_gap_samples",
        "n_gaps",
        "n_overlaps",
    ]
]

,sample,sample_rate_hz,timeline_class,timestamp_coverage,median_nb_samples,max_abs_gap_samples,n_gaps,n_overlaps
0,ego4d_32k,32000,continuous,1.0,1024.0,5.684342e-11,0,0
1,ego4d_44k1,44100,gap_or_overlap,1.0,1024.0,5.000000e+00,0,3
2,ego4d_48k,48000,continuous,1.0,1024.0,8.526513e-11,0,0
3,egocom_44k1,44100,continuous,1.0,1024.0,7.833734e-11,0,0


In [102]:
def audit_sampled_audio_timeline(
    row,
    *,
    raw_root: Path,
    window_sec: float = 10.0,
) -> list[dict]:
    path = raw_root / row.relative_path

    windows = make_timeline_windows(
        float(row.audio_duration_sec),
        window_sec=window_sec,
    )

    records = []

    for window_name, interval in windows.items():
        try:
            frames = probe_audio_frames(
                path,
                read_intervals=interval,
            )

            analysis = analyze_audio_timeline(
                frames,
                sample_rate_hz=int(row.audio_sample_rate_hz),
                time_base=row.audio_time_base,
            )

            records.append(
                {
                    "dataset": row.dataset,
                    "relative_path": row.relative_path,
                    "window": window_name,
                    "requested_interval": interval,
                    "probe_ok": True,
                    "probe_error": None,
                    **analysis,
                }
            )

        except Exception as exc:
            records.append(
                {
                    "dataset": row.dataset,
                    "relative_path": row.relative_path,
                    "window": window_name,
                    "requested_interval": interval,
                    "probe_ok": False,
                    "probe_error": str(exc),
                    "timeline_class": "probe_error",
                }
            )

    return records

In [103]:
from tqdm.auto import tqdm

audio_records = []

for row in tqdm(
    media_metadata.itertuples(index=False),
    total=len(media_metadata),
    desc="Audio timeline audit",
):
    audio_records.extend(
        audit_sampled_audio_timeline(
            row,
            raw_root=raw_root,
        )
    )

audio_timeline_samples = pd.DataFrame(audio_records)

Audio timeline audit: 100%|██████████| 369/369 [02:07<00:00,  2.88it/s]


In [104]:
audio_timeline_samples.shape

(1103, 19)

In [105]:
audio_timeline_samples["probe_ok"].value_counts(dropna=False)

probe_ok
True    1103
Name: count, dtype: int64

In [106]:
(
    audio_timeline_samples.groupby(
        ["dataset", "timeline_class"],
        dropna=False,
    )
    .size()
    .rename("n_windows")
)

dataset  timeline_class
ego4d    continuous        439
         gap_or_overlap    143
egocom   continuous        521
Name: n_windows, dtype: int64

In [107]:
audio_timeline_samples.groupby("dataset")[
    [
        "timestamp_coverage",
        "sample_count_coverage",
        "max_abs_gap_samples",
        "n_gaps",
        "n_overlaps",
    ]
].describe().T

dataset                             ego4d        egocom
timestamp_coverage    count  5.820000e+02  5.210000e+02
                      mean   1.000000e+00  1.000000e+00
                      std    0.000000e+00  0.000000e+00
                      min    1.000000e+00  1.000000e+00
                      25%    1.000000e+00  1.000000e+00
                      50%    1.000000e+00  1.000000e+00
                      75%    1.000000e+00  1.000000e+00
                      max    1.000000e+00  1.000000e+00
sample_count_coverage count  5.820000e+02  5.210000e+02
                      mean   1.000000e+00  1.000000e+00
                      std    0.000000e+00  0.000000e+00
                      min    1.000000e+00  1.000000e+00
                      25%    1.000000e+00  1.000000e+00
                      50%    1.000000e+00  1.000000e+00
                      75%    1.000000e+00  1.000000e+00
                      max    1.000000e+00  1.000000e+00
max_abs_gap_samples   count  5.820000e+02  5.210000e+02
                      mean   1.223351e+02  3.105827e-09
                      std    4.019749e+02  3.561736e-09
                      min    5.684342e-11  3.916867e-11
                      25%    8.526513e-11  7.833734e-11
                      50%    5.456968e-09  1.253397e-09
                      75%    4.365575e-08  5.013590e-09
                      max    4.753000e+03  1.002718e-08
n_gaps                count  5.820000e+02  5.210000e+02
                      mean   2.180412e+01  0.000000e+00
                      std    6.568539e+01  0.000000e+00
                      min    0.000000e+00  0.000000e+00
                      25%    0.000000e+00  0.000000e+00
                      50%    0.000000e+00  0.000000e+00
                      75%    0.000000e+00  0.000000e+00
                      max    2.380000e+02  0.000000e+00
n_overlaps            count  5.820000e+02  5.210000e+02
                      mean   2.825773e+01  0.000000e+00
                      std    8.186721e+01  0.000000e+00
                      min    0.000000e+00  0.000000e+00
                      25%    0.000000e+00  0.000000e+00
                      50%    0.000000e+00  0.000000e+00
                      75%    0.000000e+00  0.000000e+00
                      max    3.560000e+02  0.000000e+00

In [108]:
audio_file_summary = audio_timeline_samples.groupby(
    ["dataset", "relative_path"],
    as_index=False,
).agg(
    n_windows=("window", "count"),
    all_probes_ok=(
        "probe_ok",
        "all",
    ),
    all_windows_continuous=(
        "timeline_class",
        lambda s: bool((s == "continuous").all()),
    ),
    min_timestamp_coverage=(
        "timestamp_coverage",
        "min",
    ),
    min_sample_count_coverage=(
        "sample_count_coverage",
        "min",
    ),
    max_abs_gap_samples=(
        "max_abs_gap_samples",
        "max",
    ),
    total_gaps=(
        "n_gaps",
        "sum",
    ),
    total_overlaps=(
        "n_overlaps",
        "sum",
    ),
)

In [109]:
audio_file_summary.groupby(
    [
        "dataset",
        "all_windows_continuous",
    ]
).size()

dataset  all_windows_continuous
ego4d    False                      70
         True                      124
egocom   True                      175
dtype: int64

In [110]:
audio_suspects = audio_file_summary.loc[~audio_file_summary["all_windows_continuous"]]

audio_suspects

,dataset,relative_path,n_windows,all_probes_ok,all_windows_continuous,min_timestamp_coverage,min_sample_count_coverage,max_abs_gap_samples,total_gaps,total_overlaps
8,ego4d,Ego4D/v2/video_540ss/07d824bc-a3fd-4acd-8179-7...,3,True,False,1.0,1.0,1023.0,0,4
9,ego4d,Ego4D/v2/video_540ss/08920cde-46d5-4bfd-b664-1...,3,True,False,1.0,1.0,1023.0,0,4
12,ego4d,Ego4D/v2/video_540ss/0bd23c6f-d061-4fb6-95cb-b...,3,True,False,1.0,1.0,4.0,236,231
15,ego4d,Ego4D/v2/video_540ss/14c123f4-06ab-4b51-aa26-9...,3,True,False,1.0,1.0,73.0,462,474
20,ego4d,Ego4D/v2/video_540ss/19ad4939-157c-4816-acea-7...,3,True,False,1.0,1.0,216.0,235,234
...,...,...,...,...,...,...,...,...,...,...
176,ego4d,Ego4D/v2/video_540ss/e851ba96-d1eb-482c-bc5f-d...,3,True,False,1.0,1.0,1023.0,0,4
179,ego4d,Ego4D/v2/video_540ss/eb81442c-a322-49ea-b243-a...,3,True,False,1.0,1.0,1023.0,0,4
187,ego4d,Ego4D/v2/video_540ss/f7996d83-df45-4dbb-b376-f...,3,True,False,1.0,1.0,1023.0,0,4
191,ego4d,Ego4D/v2/video_540ss/fa02dd8c-03d9-4b59-82a5-c...,3,True,False,1.0,1.0,529.0,448,488


In [111]:
def probe_audio_packets(path: Path) -> list[dict]:
    command = [
        str(ffprobe_path),
        "-v",
        "error",
        "-select_streams",
        "a:0",
        "-show_packets",
        "-show_entries",
        (
            "packet=pts,pts_time,duration,duration_time:"
            "packet_side_data="
            "side_data_type,"
            "skip_samples,"
            "discard_padding,"
            "skip_reason,"
            "discard_reason"
        ),
        "-of",
        "json",
        str(path),
    ]

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        check=False,
    )

    if result.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {path}\n{result.stderr.strip()}")

    return json.loads(result.stdout).get("packets", [])

In [112]:
def extract_skip_samples(packets: list[dict]) -> pd.DataFrame:
    records = []

    for packet_index, packet in enumerate(packets):
        for side_data in packet.get(
            "side_data_list",
            [],
        ):
            if side_data.get("side_data_type") != "Skip Samples":
                continue

            records.append(
                {
                    "packet_index": packet_index,
                    "pts": packet.get("pts"),
                    "pts_time": packet.get("pts_time"),
                    "skip_samples": int(
                        side_data.get(
                            "skip_samples",
                            0,
                        )
                    ),
                    "discard_padding": int(
                        side_data.get(
                            "discard_padding",
                            0,
                        )
                    ),
                    "skip_reason": side_data.get("skip_reason"),
                    "discard_reason": side_data.get("discard_reason"),
                }
            )

    return pd.DataFrame(records)

In [114]:
skip_results = {}

for name, row in audio_samples.items():
    path = raw_root / row.relative_path

    packets = probe_audio_packets(path)
    skip = extract_skip_samples(packets)

    skip_results[name] = skip

    print(
        "\n",
        name,
        "n_packets=",
        len(packets),
    )

    display(skip)


 ego4d_32k n_packets= 40114


,packet_index,pts,pts_time,skip_samples,discard_padding,skip_reason,discard_reason
0,0,-2048,-0.064000,2048,0,0,0



 ego4d_44k1 n_packets= 274705


,packet_index,pts,pts_time,skip_samples,discard_padding,skip_reason,discard_reason
0,0,-2048,-0.046440,2048,0,0,0



 ego4d_48k n_packets= 67845


,packet_index,pts,pts_time,skip_samples,discard_padding,skip_reason,discard_reason
0,0,-2048,-0.042667,2048,0,0,0



 egocom_44k1 n_packets= 53119


,packet_index,pts,pts_time,skip_samples,discard_padding,skip_reason,discard_reason
0,0,-1024,-0.023220,1024,0,0,0


In [115]:
audio_timeline_samples.groupby(
    ["dataset", "timeline_class"],
    dropna=False,
).size()

dataset  timeline_class
ego4d    continuous        439
         gap_or_overlap    143
egocom   continuous        521
dtype: int64

In [116]:
audio_timeline_samples["probe_ok"].value_counts(dropna=False)

probe_ok
True    1103
Name: count, dtype: int64

In [117]:
audio_file_summary.groupby(["dataset", "all_windows_continuous"]).size()

dataset  all_windows_continuous
ego4d    False                      70
         True                      124
egocom   True                      175
dtype: int64

In [118]:
audio_suspects = audio_file_summary.loc[~audio_file_summary["all_windows_continuous"]]

audio_suspects[
    [
        "dataset",
        "relative_path",
        "min_timestamp_coverage",
        "min_sample_count_coverage",
        "max_abs_gap_samples",
        "total_gaps",
        "total_overlaps",
    ]
]

,dataset,relative_path,min_timestamp_coverage,min_sample_count_coverage,max_abs_gap_samples,total_gaps,total_overlaps
8,ego4d,Ego4D/v2/video_540ss/07d824bc-a3fd-4acd-8179-7...,1.0,1.0,1023.0,0,4
9,ego4d,Ego4D/v2/video_540ss/08920cde-46d5-4bfd-b664-1...,1.0,1.0,1023.0,0,4
12,ego4d,Ego4D/v2/video_540ss/0bd23c6f-d061-4fb6-95cb-b...,1.0,1.0,4.0,236,231
15,ego4d,Ego4D/v2/video_540ss/14c123f4-06ab-4b51-aa26-9...,1.0,1.0,73.0,462,474
20,ego4d,Ego4D/v2/video_540ss/19ad4939-157c-4816-acea-7...,1.0,1.0,216.0,235,234
...,...,...,...,...,...,...,...
176,ego4d,Ego4D/v2/video_540ss/e851ba96-d1eb-482c-bc5f-d...,1.0,1.0,1023.0,0,4
179,ego4d,Ego4D/v2/video_540ss/eb81442c-a322-49ea-b243-a...,1.0,1.0,1023.0,0,4
187,ego4d,Ego4D/v2/video_540ss/f7996d83-df45-4dbb-b376-f...,1.0,1.0,1023.0,0,4
191,ego4d,Ego4D/v2/video_540ss/fa02dd8c-03d9-4b59-82a5-c...,1.0,1.0,529.0,448,488


In [119]:
duration_extremes = (
    media_metadata.assign(
        abs_av_duration_delta_sec=lambda df: df["av_duration_delta_sec"].abs()
    )
    .sort_values(
        "abs_av_duration_delta_sec",
        ascending=False,
    )
    .groupby("dataset")
    .head(3)
)

duration_extremes[
    [
        "dataset",
        "relative_path",
        "audio_sample_rate_hz",
        "video_duration_sec",
        "audio_duration_sec",
        "av_duration_delta_sec",
    ]
]

,dataset,relative_path,audio_sample_rate_hz,video_duration_sec,audio_duration_sec,av_duration_delta_sec
44,ego4d,Ego4D/v2/video_540ss/3a53bfdc-daf7-4bbc-bb9f-a...,32000,1888.466667,1887.766000,-0.700667
165,ego4d,Ego4D/v2/video_540ss/cebb6331-2a18-454a-862b-3...,32000,1888.500000,1887.832000,-0.668000
83,ego4d,Ego4D/v2/video_540ss/6ec6a21e-e99c-425e-af59-6...,32000,1902.766667,1903.356000,0.589333
296,egocom,EgoCom/240p/5min_parts/vid_015__day_1__con_1__...,44100,82.533333,82.227007,-0.306326
340,egocom,EgoCom/240p/5min_parts/vid_059__day_1__con_5__...,44100,4.800000,4.528005,-0.271995
324,egocom,EgoCom/240p/5min_parts/vid_043__day_1__con_4__...,44100,290.366667,290.106009,-0.260658


In [120]:
b3_validation_paths = {
    # Quelques overlaps seulement, ~1 frame AAC.
    "boundary_like": ("Ego4D/v2/video_540ss/07d824bc-a3fd-4acd-8179-75a7e1e077ce.mp4"),
    # Beaucoup de micro-variations, max ~4 samples.
    "micro_jitter": ("Ego4D/v2/video_540ss/0bd23c6f-d061-4fb6-95cb-be013d55eace.mp4"),
    # Plus grosse discontinuité observée.
    "large_gap": ("Ego4D/v2/video_540ss/67cb31ac-4f88-4765-8806-d93276eda28a.mp4"),
}

In [121]:
b3_validation = []

for case, relative_path in b3_validation_paths.items():
    row = media_metadata.loc[media_metadata["relative_path"] == relative_path].iloc[0]

    frames = probe_audio_frames(raw_root / relative_path)

    result = analyze_audio_timeline(
        frames,
        sample_rate_hz=int(row.audio_sample_rate_hz),
        time_base=row.audio_time_base,
    )

    b3_validation.append(
        {
            "case": case,
            "relative_path": relative_path,
            "sample_rate_hz": row.audio_sample_rate_hz,
            **result,
        }
    )

pd.DataFrame(b3_validation)[
    [
        "case",
        "timeline_class",
        "n_frames",
        "timestamp_coverage",
        "max_abs_gap_samples",
        "n_gaps",
        "n_overlaps",
    ]
]

,case,timeline_class,n_frames,timestamp_coverage,max_abs_gap_samples,n_gaps,n_overlaps
0,boundary_like,gap_or_overlap,56258,1.0,1023.0,0,12
1,micro_jitter,gap_or_overlap,47195,1.0,5.0,1245,1237
2,large_gap,gap_or_overlap,69567,1.0,4753.0,10371,10604


In [122]:
def effective_audio_bounds(
    frames: pd.DataFrame,
    *,
    sample_rate_hz: int,
    time_base: str,
) -> tuple[float, float]:
    pts = pd.to_numeric(
        frames.get("pts", pd.Series(dtype=float)),
        errors="coerce",
    )

    nb_samples = pd.to_numeric(
        frames.get("nb_samples", pd.Series(dtype=float)),
        errors="coerce",
    )

    valid = pts.notna() & nb_samples.notna()

    pts = pts[valid]
    nb_samples = nb_samples[valid]

    if pts.empty:
        return np.nan, np.nan

    tb = float(Fraction(time_base))

    starts = pts * tb
    ends = starts + nb_samples / sample_rate_hz

    return float(starts.min()), float(ends.max())

In [123]:
def effective_video_bounds(
    packets: pd.DataFrame,
    *,
    fps: float,
    time_base: str,
) -> tuple[float, float]:
    pts = pd.to_numeric(
        packets.get("pts", pd.Series(dtype=float)),
        errors="coerce",
    ).dropna()

    if pts.empty:
        return np.nan, np.nan

    tb = float(Fraction(time_base))

    starts = pts * tb

    return (
        float(starts.min()),
        float(starts.max() + 1.0 / fps),
    )

In [124]:
b4_samples = {}


def add_b4_sample(name: str, mask: pd.Series) -> None:
    rows = media_metadata.loc[mask]

    if not rows.empty:
        b4_samples[name] = rows.iloc[0]


add_b4_sample(
    "egocom_normal",
    (media_metadata["dataset"] == "egocom")
    & (media_metadata["video_avg_frame_rate"] == 30)
    & (media_metadata["audio_sample_rate_hz"] == 44100),
)

add_b4_sample(
    "ego4d_32k",
    (media_metadata["dataset"] == "ego4d")
    & (media_metadata["audio_sample_rate_hz"] == 32000),
)

add_b4_sample(
    "ego4d_44k1",
    (media_metadata["dataset"] == "ego4d")
    & (media_metadata["audio_sample_rate_hz"] == 44100),
)

add_b4_sample(
    "ego4d_48k",
    (media_metadata["dataset"] == "ego4d")
    & (media_metadata["audio_sample_rate_hz"] == 48000),
)

add_b4_sample(
    "egocom_60fps",
    (media_metadata["dataset"] == "egocom")
    & (media_metadata["video_avg_frame_rate"] == 60),
)

In [ ]:
for name, relative_path in b3_validation_paths.items():
    rows = media_metadata.loc[media_metadata["relative_path"] == relative_path]

    if not rows.empty:
        b4_samples[name] = rows.iloc[0]

In [ ]:
extreme_rows = media_metadata.assign(
    abs_av_duration_delta_sec=media_metadata["av_duration_delta_sec"].abs()
).sort_values(
    "abs_av_duration_delta_sec",
    ascending=False,
)

for i, row in enumerate(
    extreme_rows.itertuples(index=False),
    start=1,
):
    if row.relative_path in {sample.relative_path for sample in b4_samples.values()}:
        continue

    b4_samples[f"duration_extreme_{i}"] = row

    if sum(name.startswith("duration_extreme_") for name in b4_samples) == 2:
        break

In [ ]:
b4_records = []

for name, row in b4_samples.items():
    path = raw_root / row.relative_path

    video_packets = probe_video_timestamps(path)
    audio_frames = probe_audio_frames(path)

    video_start, video_end = effective_video_bounds(
        video_packets,
        fps=float(row.video_avg_frame_rate),
        time_base=row.video_time_base,
    )

    audio_start, audio_end = effective_audio_bounds(
        audio_frames,
        sample_rate_hz=int(row.audio_sample_rate_hz),
        time_base=row.audio_time_base,
    )

    b4_records.append(
        {
            "sample": name,
            "dataset": row.dataset,
            "video_start_sec": video_start,
            "audio_start_sec": audio_start,
            "start_offset_sec": audio_start - video_start,
            "video_end_sec": video_end,
            "audio_end_sec": audio_end,
            "end_offset_sec": audio_end - video_end,
        }
    )

b4 = pd.DataFrame(b4_records)

b4

In [126]:
b4_sync = media_metadata.assign(
    av_start_offset_sec=(
        media_metadata["audio_start_time_sec"] - media_metadata["video_start_time_sec"]
    ),
    av_end_delta_sec=(
        media_metadata["audio_duration_sec"] - media_metadata["video_duration_sec"]
    ),
)

b4_sync.groupby("dataset")[
    [
        "av_start_offset_sec",
        "av_end_delta_sec",
    ]
].describe().T

dataset                         ego4d      egocom
av_start_offset_sec count  194.000000  175.000000
                    mean     0.000000    0.000000
                    std      0.000000    0.000000
                    min      0.000000    0.000000
                    25%      0.000000    0.000000
                    50%      0.000000    0.000000
                    75%      0.000000    0.000000
                    max      0.000000    0.000000
av_end_delta_sec    count  194.000000  175.000000
                    mean     0.129698   -0.040280
                    std      0.273435    0.066028
                    min     -0.700667   -0.306326
                    25%     -0.102667   -0.032336
                    50%     -0.001333   -0.019342
                    75%      0.437667   -0.003674
                    max      0.589333    0.029342

In [127]:
b4_sync["av_start_offset_sec"].value_counts(dropna=False)

av_start_offset_sec
0.0    369
Name: count, dtype: int64

In [128]:
b4_sync.loc[b4_sync["av_start_offset_sec"].abs() > 0][
    [
        "dataset",
        "relative_path",
        "video_start_time_sec",
        "audio_start_time_sec",
        "av_start_offset_sec",
    ]
]

,dataset,relative_path,video_start_time_sec,audio_start_time_sec,av_start_offset_sec


In [129]:
temporal_contract = {
    "video_clock": "native_pts",
    "audio_clock": "native_pts",
    "annotation_clock": "source_timestamps_to_validate_in_C",
    "raw_media_mutable": False,
    "reconstruct_video_time_from_frame_index": False,
    "reconstruct_audio_time_from_sample_index": False,
    "apply_global_av_offset": False,
    "audio_gaps": "preserve_and_mask",
    "temporal_interpolation": "defer_to_representation_stage",
    "resampling_stage": "G",
    "cross_view_sync": "deferred",
    "perceptual_av_sync": "deferred",
}

In [130]:
pd.Series(
    temporal_contract,
    name="decision",
).to_frame()

,decision
video_clock,native_pts
audio_clock,native_pts
annotation_clock,source_timestamps_to_validate_in_C
raw_media_mutable,False
reconstruct_video_time_from_frame_index,False
reconstruct_audio_time_from_sample_index,False
apply_global_av_offset,False
audio_gaps,preserve_and_mask
temporal_interpolation,defer_to_representation_stage
resampling_stage,G
